# 01. Data Assembly

Loads the Federal Reserve's 2026 severely adverse scenario and the historical
actuals for the same variables, joins them at the 2025 Q4 anchor point, and
restricts the projection window to the nine-quarter horizon.

Verifies that the resulting peak-to-trough declines in the house price index
and the commercial real estate price index match the Federal Reserve's stated
scenario severity.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

raw = Path("..")/"data"/"raw"/"fed_scenarios"
processed = Path("..")/"data"/"processed"

scenario_file = raw/"2026_Final_Supervisory_Severely_Adverse_Domestic.csv"
historic_file = raw/"2026_Final_Historic_Domestic.csv"

scenario = pd.read_csv(scenario_file)
historic = pd.read_csv(historic_file)

print(f"scenario: {scenario.shape[0]} rows, {scenario.shape[1]} columns")
print(f"historic: {historic.shape[0]} rows, {historic.shape[1]} columns")

scenario: 13 rows, 18 columns
historic: 200 rows, 18 columns


In [2]:
print(list(scenario.columns) == list(historic.columns))

True


In [3]:
scenario["Date"] = pd.PeriodIndex(scenario["Date"].str.replace(" ",""), freq="Q")
historic["Date"] = pd.PeriodIndex(historic["Date"].str.replace(" ",""), freq="Q")

print(historic["Date"].min(), "to", historic["Date"].max())
print(scenario["Date"].min(), "to", scenario["Date"].max())

1976Q1 to 2025Q4
2026Q1 to 2029Q1


In [4]:
combined = pd.concat([historic, scenario], ignore_index=True)
combined = combined.sort_values("Date").reset_index(drop=True)

print(combined.shape)
print(combined["Date"].min(), "to", combined["Date"].max())

(213, 18)
1976Q1 to 2029Q1


In [5]:
anchor = pd.Period("2025Q4", freq="Q")

index_vars = [
    "Dow Jones Total Stock Market Index (Level)",
    "House Price Index (Level)",
    "Commercial Real Estate Price Index (Level)",
]

verify_vars = [
    "House Price Index (Level)",
    "Commercial Real Estate Price Index (Level)",
]

def pct_col_name(level_col):
    """Map an index level column to its percent-change counterpart."""
    return level_col.replace(" (Level)", " (pct chg from 2025Q4)")

anchor_row = combined.loc[combined["Date"] == anchor, index_vars]
print(anchor_row)

     Dow Jones Total Stock Market Index (Level)  House Price Index (Level)  Commercial Real Estate Price Index (Level)
199                                     67501.5                      323.4                                       305.8


In [6]:
anchor_val = anchor_row.iloc[0]

for col in index_vars:
    pct_col = col.replace(" (Level)", " (pct chg from 2025Q4)")
    combined[pct_col] = (combined[col]/anchor_val[col] - 1) * 100

In [7]:
horizon_start = pd.Period("2026Q1", freq="Q")
horizon_end = pd.Period("2028Q1", freq="Q")

horizon = combined[
    (combined["Date"] >= horizon_start) & (combined["Date"] <= horizon_end)
].copy()

print(f"horizon: {horizon.shape[0]} quarters")

for col in verify_vars:
    pct_col = pct_col_name(col)
    trough = horizon[pct_col].min()
    trough_date = horizon.loc[horizon[pct_col].idxmin(), "Date"]
    print(f"{col}: {trough:.1f}% at {trough_date}")

horizon: 9 quarters
House Price Index (Level): -29.7% at 2027Q4
Commercial Real Estate Price Index (Level): -38.8% at 2027Q4


In [8]:
processed.mkdir(parents=True, exist_ok=True)

scenario_out = processed/"scenario_severely_adverse_2026.csv"
horizon.to_csv(scenario_out, index=False)

print(f"Written: {scenario_out}")
print(f"{horizon.shape[0]} rows, {horizon.shape[1]} columns")

Written: ..\data\processed\scenario_severely_adverse_2026.csv
9 rows, 21 columns


In [12]:
historic_out = processed/"macro_history.csv"
macro_history = combined[combined["Date"] <= anchor].copy()
macro_history.to_csv(historic_out, index=False)

print(f"Written: {historic_out}")
print(f"{macro_history.shape[0]} rows, {macro_history.shape[1]} columns")

Written: ..\data\processed\macro_history.csv
200 rows, 21 columns


## Summary

Scenario and historical macro variables joined at the 2025 Q4 anchor.

**Verification.** Peak-to-trough declines over the nine-quarter horizon,
measured from the 2025 Q4 anchor:

| Variable | Decline | Trough |
|---|---|---|
| House Price Index | -29.7% | 2027 Q4 |
| Commercial Real Estate Price Index | -38.8% | 2027 Q4 |

These match the Federal Reserve's stated scenario severity of approximately
30 percent and 39 percent respectively, confirming that the anchor point and
horizon filter are correctly applied.

**Outputs.**

| File | Contents | Rows |
|---|---|---|
| `data/processed/scenario_severely_adverse_2026.csv` | Projection window, 2026 Q1 to 2028 Q1 | 9 |
| `data/processed/macro_history.csv` | Historical actuals, 1976 Q1 to 2025 Q4 | 200 |

In [16]:
from pathlib import Path
import pandas as pd

raw_y9c = Path("..") / "data" / "raw" / "y9c"
y9c = pd.read_csv(raw_y9c / "FRY9C_1037003_20251231.csv")

print(y9c.shape)
print(y9c.head())


(1647, 3)
           ItemName Description                 Value
0  Institution Name         NaN  M&T BANK CORPORATION
1    Street Address         NaN         ONE M&T PLAZA
2              City         NaN               BUFFALO
3             State         NaN                    NY
4          Zip Code         NaN             142032399


In [17]:
def find(keyword):
    """Search the Description column for a keyword."""
    mask = y9c["Description"].str.contains(keyword, case=False, na=False)
    return y9c.loc[mask, ["ItemName", "Description", "Value"]]

In [20]:
print(find("COMMON EQUITY TIER 1"))

     ItemName                                        Description     Value
918  BHCAP839  COMMON EQUITY TIER 1 MINORITY INTEREST INCLUDA...         0
919  BHCAP840  COMMON EQUITY TIER 1 CAPITAL BEFORE ADJUSTMENT...  26342751
923  BHCAQ258  OTHER DEDUCTIONS FROM (ADDITIONS TO) COMMON EQ...         0
924  BHCAP850  OTHER DEDUCTIONS FROM (ADDITIONS TO) COMMON EQ...         0
925  BHCAP852  SUBTOTAL OF COMMON EQUITY TIER 1 CAPITAL (ITEM...  17551365
926  BHCAP857  DEDUCTIONS APPLIED TO COMMON EQUITY TIER 1 CAP...         0
927  BHCAP858  TOTAL ADJUSTMENTS AND DEDUCTIONS FOR COMMON EQ...         0
928  BHCAP859  COMMON EQUITY TIER 1 CAPITAL (ITEM 12 MINUS IT...  17551365
931  BHCAP862  TIER 1 MINORITY INTEREST NOT INCLUDED IN COMMO...         0
944  BHCAP875  DEDUCTIONS FROM COMMON EQUITY TIER 1 CAPITAL A...   8508890
948  BHCAP793  COMMON EQUITY TIER 1 CAPITAL RATIO (ITEM 19 DI...   10.8414


In [21]:
print(find("RISK-WEIGHTED ASSETS"))

      ItemName                                        Description      Value
763   BHCKB704  RISK-WEIGHTED ASSETS BEFORE DEDUCTIONS FOR EXC...  162024059
947   BHCAA223                         TOTAL RISK-WEIGHTED ASSETS  161891600
1423  BHCKG634  RISK-WEIGHTED ASSETS BY RISK-WEIGHT CATEGORY (...          0
1424  BHCKS569  RISK-WEIGHTED ASSETS BY RISK-WEIGHT CATEGORY (...          0
1425  BHCKS570  RISK-WEIGHTED ASSETS BY RISK-WEIGHT CATEGORY (...        342
1426  BHCKS571  RISK-WEIGHTED ASSETS BY RISK-WEIGHT CATEGORY (...          0
1427  BHCKG635  RISK-WEIGHTED ASSETS BY RISK-WEIGHT CATEGORY (...    5239889
1428  BHCKG636  RISK-WEIGHTED ASSETS BY RISK-WEIGHT CATEGORY (...   13199886
1429  BHCKG637  RISK-WEIGHTED ASSETS BY RISK-WEIGHT CATEGORY (...  140380686
1430  BHCKS572  RISK-WEIGHTED ASSETS BY RISK-WEIGHT CATEGORY (...    1457327
1431  BHCKS574  RISK-WEIGHTED ASSETS BY RISK-WEIGHT CATEGORY (...          0
1432  BHCKS575  RISK-WEIGHTED ASSETS BY RISK-WEIGHT CATEGORY (...          0

In [23]:
print(find("LOANS SECURED BY REAL ESTATE"))

      ItemName                                        Description     Value
41    BHCK1410    LOANS SECURED BY REAL ESTATE (BHC CONSOLIDATED)  63639202
104   BHCKB837  LOANS SECURED BY REAL ESTATE TO NON-US ADDRESS...     13173
153   BHCK4652  LOANS SECURED BY REAL ESTATE, NON-U.S. ADDRESS...         0
154   BHCK4662  LOANS SECURED BY REAL ESTATE, NON-U.S. ADDRESS...         0
209   BHCKB512  LOANS SECURED BY REAL ESTATE: IN FOREIGN OFFIC...         0
210   BHCKB513  LOANS SECURED BY REAL ESTATE: IN FOREIGN OFFIC...         0
667   BHCKB572  LOANS SECURED BY REAL ESTATE, IN FOREIGN OFFIC...         0
668   BHCKB573  LOANS SECURED BY REAL ESTATE, IN FOREIGN OFFIC...         0
669   BHCKB574  LOANS SECURED BY REAL ESTATE, IN FOREIGN OFFIC...         0
1097  BHCK4436  INTEREST AND FEE INCOME ON LOANS: IN DOMESTIC ...   2188860
1105  BHDM3466  QUARTERLY AVERAGES: ALL OTHER LOANS SECURED BY...  34104041
1119  BHCKG091  LOANS (NOT CONSIDERED PURCHASED CREDIT DETERIO...         0
1120  BHCKG0

In [25]:
print(find("1-4 FAMILY"))

      ItemName                                        Description     Value
265   BHCKC234  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDEN...      4238
266   BHCKC217  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDEN...      4230
267   BHCKC235  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDEN...        22
268   BHCKC218  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDEN...      1891
312   BHCKG360  STRUCTURED FINANCIAL PRODUCTS BY UNDERLYING CO...         0
313   BHCKG361  STRUCTURED FINANCIAL PRODUCTS BY UNDERLYING CO...         0
314   BHCKG362  STRUCTURED FINANCIAL PRODUCTS BY UNDERLYING CO...         0
315   BHCKG363  STRUCTURED FINANCIAL PRODUCTS BY UNDERLYING CO...         0
316   BHCKG364  STRUCTURED FINANCIAL PRODUCTS BY UNDERLYING CO...         0
317   BHCKG365  STRUCTURED FINANCIAL PRODUCTS BY UNDERLYING CO...         0
318   BHCKG366  STRUCTURED FINANCIAL PRODUCTS BY UNDERLYING CO...         0
319   BHCKG367  STRUCTURED FINANCIAL PRODUCTS BY UNDERLYING CO...         0
673   BHCKB8

In [26]:
print(find("NONFARM NONRESIDENTIAL"))

      ItemName                                        Description     Value
792   BHDMK161  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING ...      9501
793   BHDMK162  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING ...    224233
844   BHDMK114  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING ...       784
845   BHDMK115  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING ...         0
846   BHDMK116  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING ...     25168
847   BHDMK117  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING ...     85444
848   BHDMK118  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING ...         0
849   BHDMK119  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING ...     45162
1056  BHCKC895  CHARGE-OFFS: LOANS SECURED BY OWNER-OCCUPIED N...      2000
1057  BHCKC896  RECOVERIES: LOANS SECURED BY OWNER-OCCUPIED NO...      6437
1058  BHCKC897  CHARGE-OFFS: LOANS SECURED BY OTHER NONFARM NO...     82177
1059  BHCKC898  RECOVERIES: LOANS SECURED BY OTHER NONFARM NON...     17272
1065  BHCKF1

In [28]:
pd.set_option("display.max_colwidth", 80)
print(y9c.iloc[38:75])

    ItemName                                                                      Description      Value
38  BHCK0384          DEBT SECURITIES WITH REMAINING MATURITY OF 1-5 YEARS (BHC CONSOLIDATED)   11139241
39  BHCK0387           DEBT SECURITIES WITH REMAINING MATURITY OF 5+ YEARS (BHC CONSOLIDATED)   19540357
40  BHCK0416                                            PLEDGED SECURITIES (BHC CONSOLIDATED)   20407790
41  BHCK1410                                  LOANS SECURED BY REAL ESTATE (BHC CONSOLIDATED)   63639202
42  BHCK1763  COMMERCIAL AND INDUSTRIAL LOANS TO U.S. ADDRESSEES (DOMICILE) (BHC CONSOLIDA...   32904960
43  BHCK1764  COMMERCIAL AND INDUSTRIAL LOANS TO NON-U.S. ADDRESSEES (DOMICILE) (BHC CONSO...     113884
44  BHCK1590  LOANS TO FINANCE AGRICULTURAL PRODUCTION AND OTHER LOANS TO FARMERS (BHC CON...      72707
45  BHCK2081        LOANS TO FOREIGN GOVERNMENTS AND OFFICIAL INSTITUTIONS (BHC CONSOLIDATED)          0
46  BHCK2123                                      UNEAR

In [29]:
print(find("CLOSED-END"))

      ItemName                                                                      Description    Value
265   BHCKC234  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDENTIAL PROPERTIES: SECURED BY FI...     4238
266   BHCKC217  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDENTIAL PROPERTIES: SECURED BY FI...     4230
267   BHCKC235  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDENTIAL PROPERTIES: SECURED BY JU...       22
268   BHCKC218  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDENTIAL PROPERTIES: SECURED BY JU...     1891
749   BHCKC236  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDENTIAL PROPERTIES: SECURED BY FI...   656777
750   BHCKC237  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDENTIAL PROPERTIES: SECURED BY FI...   651460
751   BHCKC229  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDENTIAL PROPERTIES: SECURED BY FI...   264439
752   BHCKC238  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDENTIAL PROPERTIES: SECURED BY JU...      706
753   BHCKC239  CLOSED-END LOANS SECURED BY 1-4 FAMILY 

In [30]:
print(y9c[y9c["ItemName"].isin(["BHDM5367", "BHDM5368", "BHDM1415", "BHDM1480"])])

    ItemName                                                  Description     Value
77  BHDM5367  ALL OTH LNS SECD BY 1-4FMLY-1ST LIEN (BHC CONSOL. U.S. OFC)  24793482
78  BHDM5368  ALL OTH LNS SECD BY 1-4FMLY-JR LIENS (BHC CONSOL. U.S. OFC)     49441


In [31]:
print(y9c.iloc[65:90])

       ItemName                                                                      Description              Value
65     BHCK3296                          VAR RT INT-BRG DEPS RPRIC OR MAT 1YR (BHC CONSOLIDATED)           12801077
66     BHCK3408                                 VARIABLE RATE PREFERRED STOCK (BHC CONSOLIDATED)                  0
67     BHCK3409  LONG-TERM DEBT REPORTED IN SCHDULE HC, ITEM 19A ON THE BALANCE SHEET THAT IS...                  0
68     BHCK3368                                      QTLY AVG OF TOTAL ASSETS (BHC CONSOLIDATED)          212739023
69     BHCK3517                          AVERAGE BALS OF INT-BEARING DEPOSITS (BHC CONSOLIDATED)          120756015
70     BHCK3404                          QTLY AVG OF INT-BRG DEPS IN FGN OFFC (BHC CONSOLIDATED)                  0
71     BHCK3519                                AVERAGE BALS OF EQUITY CAPITAL (BHC CONSOLIDATED)           28968846
72     BHCK2635                          ALL OTHER BORROWED MONEY (QTRLY

In [32]:
print(find("NONFARM"))

      ItemName                                                                      Description     Value
792   BHDMK161  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY THAT ARE ...      9501
793   BHDMK162  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY THAT ARE ...    224233
844   BHDMK114  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY INCLUDED ...       784
845   BHDMK115  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY INCLUDED ...         0
846   BHDMK116  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY INCLUDED ...     25168
847   BHDMK117  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY INCLUDED ...     85444
848   BHDMK118  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY INCLUDED ...         0
849   BHDMK119  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY INCLUDED ...     45162
1056  BHCKC895  CHARGE-OFFS: LOANS SECURED BY 

In [33]:
components = {
    "BHDM5367": "1-4 family first lien",
    "BHDM5368": "1-4 family junior lien",
    "BHDM1797": "Home equity revolving",
    "BHDM1460": "Multifamily",
    "BHCK2746": "Construction and land development",
    "BHDM1420": "Farmland",
    "BHCKF160": "CRE owner-occupied",
    "BHCKF161": "CRE other",
}

subset = y9c[y9c["ItemName"].isin(components.keys())].copy()
subset["Value"] = pd.to_numeric(subset["Value"])
subset["Label"] = subset["ItemName"].map(components)

print(subset[["ItemName", "Label", "Value"]].to_string(index=False))
print()

total_components = subset["Value"].sum()
total_reported = pd.to_numeric(
    y9c.loc[y9c["ItemName"] == "BHCK1410", "Value"].iloc[0]
)

print(f"Sum of components: {total_components:,}")
print(f"BHCK1410 reported: {total_reported:,}")
print(f"Difference:        {total_components - total_reported:,}")

ItemName                             Label    Value
BHDM1420                          Farmland   125808
BHDM1797             Home equity revolving  4862040
BHDM1460                       Multifamily  6840992
BHCK2746 Construction and land development  4535322
BHDM5367             1-4 family first lien 24793482
BHDM5368            1-4 family junior lien    49441
BHCKF160                CRE owner-occupied 10386357
BHCKF161                         CRE other 12861910

Sum of components: 64,455,352
BHCK1410 reported: 63,639,202
Difference:        816,150


In [34]:
print(find("CONSTR"))

      ItemName                                                                      Description    Value
57    BHCK2746                          LNS TO FINC COMMCL RE,CONSTRC&LD DEV (BHC CONSOLIDATED)  4535322
789   BHDMK158  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY DEVELOPME...     3660
790   BHDMK159  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY THAT ARE ...   305303
835   BHDMK105  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY INCLUDED ...        0
836   BHDMK106  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY INCLUDED ...        0
837   BHDMK107  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY INCLUDED ...        0
838   BHDMK108  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY INCLUDED ...        0
839   BHDMK109  LOANS MODIFICATIONS TO BORROWERS EXPERIENCING FINANCIAL DIFFICULTY INCLUDED ...        0
840   BHDMK110  LOANS MODIFICATIONS TO BORROWERS EXPERI

In [35]:
components = {
    "BHDM5367": "1-4 family first lien",
    "BHDM5368": "1-4 family junior lien",
    "BHDM1797": "Home equity revolving",
    "BHDM1460": "Multifamily",
    "BHCKF158": "1-4 family residential construction",
    "BHCKF159": "Other construction and land development",
    "BHDM1420": "Farmland",
    "BHCKF160": "CRE owner-occupied",
    "BHCKF161": "CRE other",
}

subset = y9c[y9c["ItemName"].isin(components.keys())].copy()
subset["Value"] = pd.to_numeric(subset["Value"])
subset["Label"] = subset["ItemName"].map(components)

print(subset[["ItemName", "Label", "Value"]].to_string(index=False))
print()

total_components = subset["Value"].sum()
total_reported = pd.to_numeric(
    y9c.loc[y9c["ItemName"] == "BHCK1410", "Value"].iloc[0]
)

print(f"Sum of components: {total_components:,}")
print(f"BHCK1410 reported: {total_reported:,}")
print(f"Difference:        {total_components - total_reported:,}")

ItemName                                   Label    Value
BHDM1420                                Farmland   125808
BHDM1797                   Home equity revolving  4862040
BHDM1460                             Multifamily  6840992
BHDM5367                   1-4 family first lien 24793482
BHDM5368                  1-4 family junior lien    49441
BHCKF158     1-4 family residential construction   219788
BHCKF159 Other construction and land development  3499384
BHCKF160                      CRE owner-occupied 10386357
BHCKF161                               CRE other 12861910

Sum of components: 63,639,202
BHCK1410 reported: 63,639,202
Difference:        0


In [36]:
print(find("CREDIT CARD"))

      ItemName                                                                      Description     Value
102   BHCKB538  LOANS TO INDIVIDUALS FOR HOUSEHOLD, FAMILY, AND OTHER PERSONAL EXPENDITURES,...    972663
211   BHCKB514  LOANS TO INDIVIDUALS FOR HOUSEHOLD, FAMILY, AND OTHER PERSONAL EXPENDITURES:...     46217
212   BHCKB515  LOANS TO INDIVIDUALS FOR HOUSEHOLD, FAMILY, AND OTHER PERSONAL EXPENDITURES:...      7601
218   BHCKB838  ASSET-BACKED SECURITIES: CREDIT CARD RECEIVABLES, H-T-M AMORTIZED COST (BHC ...         0
219   BHCKB839  ASSET-BACKED SECURITIES: CREDIT CARD RECEIVABLES, H-T-M FAIR VALUE (BHC CONS...         0
220   BHCKB840  ASSET-BACKED SECURITIES: CREDIT CARD RECEIVABLES, A-F-S AMORTIZED COST (BHC ...         0
221   BHCKB841  ASSET-BACKED SECURITIES: CREDIT CARD RECEIVABLES A-F-S FAIR VALUE (BHC CONSO...         0
521   BHCKJ455                UNUSED COMMITMENTS: CONSUMER CREDIT CARD LINES (BHC CONSOLIDATED)   4978559
522   BHCKJ456                   UNUSED COMMIT

In [37]:
print(y9c.iloc[95:115])

     ItemName                                                                      Description      Value
95   BHCK1296                                        LOANS TO FOREIGN BANKS (BHC CONSOLIDATED)          0
96   BHFNA245  FOREIGN OFFICE TIME DEPOSITS WITH A REMAINING MATURITY OF ONE YEAR OR LESS (...          0
97   BHCKB528                         LOANS AND LEASES, HELD FOR INVESTMENT (BHC CONSOLIDATED)  137742340
98   BHCKB529  LOANS AND LEASES, HELD FOR INVESTMENT, NET OF ALLOWANCE FOR LOAN AND LEASE L...  135626777
99   BHCK3190                                          OTHER BORROWED MONEY (BHC CONSOLIDATED)   11370817
100  BHCKB530        EQUITY CAPITAL, ACCUMULATED OTHER COMPREHENSIVE INCOME (BHC CONSOLIDATED)     277234
101  BHCKA130               EQUITY CAPITAL, OTHER EQUITY CAPITAL COMPONENTS (BHC CONSOLIDATED)   -4916301
102  BHCKB538  LOANS TO INDIVIDUALS FOR HOUSEHOLD, FAMILY, AND OTHER PERSONAL EXPENDITURES,...     972663
103  BHCKB539  LOANS TO INDIVIDUALS FOR HOUSEH

In [39]:
print(y9c[y9c["ItemName"].isin(["BHCKK137", "BHDMK137", "BHCKB539", "BHCK2011"])])

     ItemName                                                                      Description    Value
103  BHCKB539  LOANS TO INDIVIDUALS FOR HOUSEHOLD, FAMILY, AND OTHER PERSONAL EXPENDITURES,...    61778
787  BHCKK137  LOANS TO INDIVIDUALS FOR HOUSEHOLD, FAMILY, AND OTHER PERSONAL EXPENDITURES:...  5167727


In [40]:
all_segments = {
    "BHDM5367": "RE: 1-4 family first lien",
    "BHDM5368": "RE: 1-4 family junior lien",
    "BHDM1797": "RE: Home equity revolving",
    "BHDM1460": "RE: Multifamily",
    "BHCKF158": "RE: 1-4 family construction",
    "BHCKF159": "RE: Other construction and land",
    "BHDM1420": "RE: Farmland",
    "BHCKF160": "RE: CRE owner-occupied",
    "BHCKF161": "RE: CRE other",
    "BHCK1763": "C&I: US addressees",
    "BHCK1764": "C&I: Non-US addressees",
    "BHCKB538": "Consumer: Credit card",
    "BHCKB539": "Consumer: Other revolving",
    "BHCKK137": "Consumer: Automobile",
    "BHCKK207": "Consumer: Other",
    "BHCK1590": "Agricultural",
    "BHDM2165": "Lease financing",
}

seg = y9c[y9c["ItemName"].isin(all_segments)].copy()
seg["Value"] = pd.to_numeric(seg["Value"])
seg["Label"] = seg["ItemName"].map(all_segments)

total_seg = seg["Value"].sum()
total_loans = pd.to_numeric(y9c.loc[y9c["ItemName"] == "BHCK2122", "Value"].iloc[0])

print(seg[["ItemName", "Label", "Value"]].sort_values("Label").to_string(index=False))
print(f"\nSum of segments: {total_seg:,}")
print(f"BHCK2122 total:  {total_loans:,}")
print(f"Difference:      {total_seg - total_loans:,}")

ItemName                           Label    Value
BHCK1590                    Agricultural    72707
BHCK1764          C&I: Non-US addressees   113884
BHCK1763              C&I: US addressees 32904960
BHCKK137            Consumer: Automobile  5167727
BHCKB538           Consumer: Credit card   972663
BHCKK207                 Consumer: Other 14798677
BHCKB539       Consumer: Other revolving    61778
BHDM2165                 Lease financing  2747135
BHCKF158     RE: 1-4 family construction   219788
BHDM5367       RE: 1-4 family first lien 24793482
BHDM5368      RE: 1-4 family junior lien    49441
BHCKF161                   RE: CRE other 12861910
BHCKF160          RE: CRE owner-occupied 10386357
BHDM1420                    RE: Farmland   125808
BHDM1797       RE: Home equity revolving  4862040
BHDM1460                 RE: Multifamily  6840992
BHCKF159 RE: Other construction and land  3499384

Sum of segments: 120,478,733
BHCK2122 total:  138,809,737
Difference:      -18,331,004


In [41]:
print(find("DEPOSITORY INSTITUTIONS"))

      ItemName                                                                      Description     Value
7     BHCK0081  NONINTEREST BEARING BALANCES AND CURRENCY AND COIN DUE FROM DEPOSITORY INSTI...   1701389
93    BHDM1288                          LOANS TO DEPOSITORY INSTITUTIONS (BHC CONSOL. U.S. OFC)      6287
94    BHCK1292    LOANS TO U.S. BANKS AND OTHER U.S. DEPOSITORY INSTITUTIONS (BHC CONSOLIDATED)      6287
194   BHCK4115  INTEREST INCOME ON BALANCES DUE FROM DEPOSITORY INSTITUTIONS (BHC CONSOLIDATED)    815593
874   BHCKJ981  ASSETS OF CONSOLIDATED VARIABLE INTEREST ENTITIES (VIES) THAT CAN BE USED TO...         0
1150  BHCKD957  CASH AND BALANCES DUE FROM DEPOSITORY INSTITUTIONS (TOTALS FROM SCHEDULE HC)...  18769198
1151  BHCKS396  CASH AND BALANCES DUE FROM DEPOSITORY INSTITUTIONS (ADJUSTMENTS TO TOTALS RE...       100
1152  BHCKD958  CASH AND BALANCES DUE FROM DEPOSITORY INSTITUTIONS ( ALLOCATION BY RISK WEIG...  18003795
1153  BHCKD959  CASH AND BALANCES DUE FROM DEP

In [42]:
print(find("ALL OTHER LOANS"))

      ItemName                                                                      Description     Value
174   BHCK4644                                   CHG-OFFS ON ALL OTHER LOANS (BHC CONSOLIDATED)     31182
175   BHCK4628                                      RECOV ON ALL OTHER LOANS (BHC CONSOLIDATED)      1924
517   BHCKJ451        ALL OTHER LOANS(EXCLUDE CONSUMER LOANS) (CONSOLIDATED) (BHC CONSOLIDATED)   3732700
518   BHDMJ451        ALL OTHER LOANS(EXCLUDE CONSUMER LOANS) (DOMESTIC) (BHC CONSOL. U.S. OFC)   3732700
570   BHCK5460                          ALL OTHER LOANS-PAST DU 90 DYS OR MO (BHC CONSOLIDATED)         0
571   BHCK5461                                    ALL OTHER LOANS-NONACCRUAL (BHC CONSOLIDATED)     47444
665   BHCK5459                            ALL OTHER LOANS-PAST DU 30-89 DAYS (BHC CONSOLIDATED)     45301
682   BHCKB711  OUTSTANDING PRINCIPLE BAL. OF ASSETS SOLD & SECURITIZED WITH SERVICING RETAI...         0
689   BHCKB732  REPORTING BANK'S UNUSED COMMIT

In [43]:
def val(code):
    return pd.to_numeric(y9c.loc[y9c["ItemName"] == code, "Value"].iloc[0])

total = val("BHCK2122")
known = {
    "Real estate (BHCK1410)": val("BHCK1410"),
    "C&I US (BHCK1763)": val("BHCK1763"),
    "C&I non-US (BHCK1764)": val("BHCK1764"),
    "Individuals (BHDM1975)": val("BHDM1975"),
    "Agricultural (BHCK1590)": val("BHCK1590"),
    "Lease financing (BHDM2165)": val("BHDM2165"),
    "Depository institutions (BHCK1292)": val("BHCK1292"),
    "All other (BHCKJ451)": val("BHCKJ451"),
}

for label, v in known.items():
    print(f"{label:40s} {v:>14,}")

print(f"{'-' * 55}")
print(f"{'Sum of known':40s} {sum(known.values()):>14,}")
print(f"{'BHCK2122 total':40s} {total:>14,}")
print(f"{'Unexplained':40s} {total - sum(known.values()):>14,}")

Real estate (BHCK1410)                       63,639,202
C&I US (BHCK1763)                            32,904,960
C&I non-US (BHCK1764)                           113,884
Individuals (BHDM1975)                       21,000,845
Agricultural (BHCK1590)                          72,707
Lease financing (BHDM2165)                    2,747,135
Depository institutions (BHCK1292)                6,287
All other (BHCKJ451)                          3,732,700
-------------------------------------------------------
Sum of known                                124,217,720
BHCK2122 total                              138,809,737
Unexplained                                  14,592,017


In [44]:
print(find("C&I"))

     ItemName                                              Description     Value
47   BHDM1766              C&I LOANS, ALL OTHER (BHC CONSOL. U.S. OFC)  32973203
155  BHCK4645    CHG-OFFS ON C&I LNS-U.S. ADDRESSES (BHC CONSOLIDATED)    282485
157  BHCK4646   CHG-OFFS ON C&I LNS-NON-U.S. ADDRES (BHC CONSOLIDATED)         0
544  BHCK1607  C&I LN PAST DUE 90 OR MORE, ACCRUING (BHC CONSOLIDATED)      5332
545  BHCK1608                 C&I LOANS, NONACCRUAL (BHC CONSOLIDATED)    338340
653  BHCK1606  C&I LN PAST DUE 30-89 DAYS, ACCRUING (BHC CONSOLIDATED)    226690


In [45]:
print(find("NONDEPOSITORY"))

      ItemName                                                                      Description    Value
1017  BHCK1545  LOANS TO NONDEPOSITORY FINANCIAL INSTITUTIONS AND OTHER LOANS: OTHER LOANS L...  2055500
1018  BHDM1545  LOANS TO NONDEPOSITORY FINANCIAL INSTITUTIONS AND OTHER LOANS: OTHER LOANS L...  2055500


In [46]:
print(find("FINANCIAL INSTITUTIONS"))

      ItemName                                                                      Description     Value
300   BHCKG348  STRUCTURED FINANCIAL PRODUCTS BY UNDERLYING COLLATERAL OR REFERENCE ASSETS: ...         0
301   BHCKG349  STRUCTURED FINANCIAL PRODUCTS BY UNDERLYING COLLATERAL OR REFERENCE ASSETS: ...         0
302   BHCKG350  STRUCTURED FINANCIAL PRODUCTS BY UNDERLYING COLLATERAL OR REFERENCE ASSETS: ...         0
303   BHCKG351  STRUCTURED FINANCIAL PRODUCTS BY UNDERLYING COLLATERAL OR REFERENCE ASSETS: ...         0
515   BHCKJ454  LOANS TO NON-DEPOSITORY FINANCIAL INSTITUTIONS (CONSOLIDATED) (BHC CONSOLIDA...  12536517
516   BHDMJ454  LOANS TO NON-DEPOSITORY FINANCIAL INSTITUTIONS (DOMESTIC) (BHC CONSOL. U.S. ...  12478028
524   BHCKJ458     OTHER UNUSED COMMITMENTS: LOANS TO FINANCIAL INSTITUTIONS (BHC CONSOLIDATED)  11431921
1017  BHCK1545  LOANS TO NONDEPOSITORY FINANCIAL INSTITUTIONS AND OTHER LOANS: OTHER LOANS L...   2055500
1018  BHDM1545  LOANS TO NONDEPOSITORY FINANCI

In [47]:
print(find("POLITICAL SUBDIVISIONS"))

      ItemName                                                                      Description    Value
205   BHCK4313  INCOME ON TAX-EXEMPT LOANS AND LEASES TO STATES AND POLITICAL SUBDIVISIONS I...   126466
214   BHCK8496  SECURITIES ISSUED BY US STATE & POLITICAL SUBDIVISIONS, H-T-M, AMORTIZED COS...  2130871
215   BHCK8497  SECURITIES ISSUED BY US STATE & POLITICAL SUBDIVISIONS, H-T-M, FAIR VALUE (B...  2088915
216   BHCK8498  SECURITIES ISSUED BY US STATE & POLITICAL SUBDIVISIONS, A-F-S, AMORTIZED COS...        0
217   BHCK8499  SECURITIES ISSUED BY US STATE & POLITICAL SUBDIVISIONS, A-F-S, FAIR VALUE (B...        0
892   BHCM3533  SECURITIES ISSUED BY STATES AND POLITICAL SUBDIVISIONS IN THE U.S. (CONSOLID...        0
1622  BHCKJJ20  ALLOWANCE BALANCE FOR HELD-TO-MATURITY SECURITIES SECURITIES ISSUED BY STATE...     2000


In [49]:
known = {
    "Real estate (BHCK1410)": val("BHCK1410"),
    "C&I US (BHCK1763)": val("BHCK1763"),
    "C&I non-US (BHCK1764)": val("BHCK1764"),
    "Individuals (BHDM1975)": val("BHDM1975"),
    "Agricultural (BHCK1590)": val("BHCK1590"),
    "Lease financing (BHDM2165)": val("BHDM2165"),
    "Depository institutions (BHCK1292)": val("BHCK1292"),
    "Nondepository financial (BHCKJ454)": val("BHCKJ454"),
    "Other loans (BHCK1545)": val("BHCK1545"),
}

for label, v in known.items():
    print(f"{label:40s} {v:>14,}")
known["All other (BHCKJ451)"] = val("BHCKJ451")
print("-" * 55)
print(f"{'Sum of known':40s} {sum(known.values()):>14,}")
print(f"{'BHCK2122 total':40s} {val('BHCK2122'):>14,}")
print(f"{'Unexplained':40s} {val('BHCK2122') - sum(known.values()):>14,}")

Real estate (BHCK1410)                       63,639,202
C&I US (BHCK1763)                            32,904,960
C&I non-US (BHCK1764)                           113,884
Individuals (BHDM1975)                       21,000,845
Agricultural (BHCK1590)                          72,707
Lease financing (BHDM2165)                    2,747,135
Depository institutions (BHCK1292)                6,287
Nondepository financial (BHCKJ454)           12,536,517
Other loans (BHCK1545)                        2,055,500
-------------------------------------------------------
Sum of known                                138,809,737
BHCK2122 total                              138,809,737
Unexplained                                           0


In [50]:
chargeoff = y9c[y9c["Description"].str.startswith(("CHARGE-OFFS", "RECOVERIES", "CHG-OFFS", "RECOV"), na=False)].copy()
chargeoff["Value"] = pd.to_numeric(chargeoff["Value"], errors="coerce")

print(f"{chargeoff.shape[0]} items")
chargeoff[["ItemName", "Description", "Value"]].to_string(index=False)

45 items


'ItemName                                                                                                                                                                                   Description  Value\nBHCK4645                                                                                                                                         CHG-OFFS ON C&I LNS-U.S. ADDRESSES (BHC CONSOLIDATED) 282485\nBHCK4617                                                                                                                                          RECOV ON C & I LNS-U.S. ADDRESSES (BHC CONSOLIDATED)  61612\nBHCK4646                                                                                                                                        CHG-OFFS ON C&I LNS-NON-U.S. ADDRES (BHC CONSOLIDATED)      0\nBHCK4618                                                                                                                                         RECOV ON C & I LNS-NON-U.S

In [51]:
print(find("CHG-OFFS"))

     ItemName                                              Description   Value
155  BHCK4645    CHG-OFFS ON C&I LNS-U.S. ADDRESSES (BHC CONSOLIDATED)  282485
157  BHCK4646   CHG-OFFS ON C&I LNS-NON-U.S. ADDRES (BHC CONSOLIDATED)       0
161  BHCK4643   CHG-OFFS ON LNS TO FGN GOVTS & INST (BHC CONSOLIDATED)       0
165  BHCK3584   CHG-OFFS ON LNS SECURED BY FARMLAND (BHC CONSOLIDATED)       0
167  BHCK3588  CHG-OFFS ON LNS SECD BY MULTI-FAMILY (BHC CONSOLIDATED)   26424
170  BHCK5411   CHG-OFFS OF RVLVG,OPEN-END LNS SECD (BHC CONSOLIDATED)    3863
172  BHCK5409  CHG-OFFS OF LNS TO FINC COMMCL RE,CO (BHC CONSOLIDATED)    1073
174  BHCK4644           CHG-OFFS ON ALL OTHER LOANS (BHC CONSOLIDATED)   31182


In [52]:
print(y9c.iloc[150:195])

     ItemName                                                                      Description   Value
150  BHCK3578         CONVERSION OR RETIREMENT OF PERPETUAL PREFERRED STOCK (BHC CONSOLIDATED)       0
151  BHCK3579                                   SALE OF COMMON STOCK, GROSS (BHC CONSOLIDATED)       0
152  BHCK3580                      CONVERSION OR RETIREMENT OF COMMON STOCK (BHC CONSOLIDATED)    8884
153  BHCK4652  LOANS SECURED BY REAL ESTATE, NON-U.S. ADDRESSES: CHARGE-OFFS (BHC CONSOLIDA...       0
154  BHCK4662  LOANS SECURED BY REAL ESTATE, NON-U.S. ADDRESSES: RECOVERIES (BHC CONSOLIDATED)       0
155  BHCK4645                            CHG-OFFS ON C&I LNS-U.S. ADDRESSES (BHC CONSOLIDATED)  282485
156  BHCK4617                             RECOV ON C & I LNS-U.S. ADDRESSES (BHC CONSOLIDATED)   61612
157  BHCK4646                           CHG-OFFS ON C&I LNS-NON-U.S. ADDRES (BHC CONSOLIDATED)       0
158  BHCK4618                            RECOV ON C & I LNS-NON-U.S. ADDR

In [53]:
print(y9c.iloc[260:290])

     ItemName                                                                      Description    Value
260  BHCK5523  WRITE-DOWNS ARISING FROM TRANSFERS OF LOANS TO THE HELD-FOR-SALE ACCOUNT (BH...        0
261  BHCKC159  DO YOUR AGGREGATE NONFINANCIAL EQUITY INVESTMENTS EQUAL OR EXCEED THE LESSER...        1
262  BHCKC161  DOES THE BHC HOLD, DIRECT OR INDIRECT, ANY NONFINANCIAL EQUITY INVESTMENTS W...        1
263  BHCKC216               NONINTEREST EXPENSE: GOODWILL IMPAIRMENT LOSSES (BHC CONSOLIDATED)        0
264  BHCKC232  NONINTEREST EXPENSE: AMORTIZATION EXPENSE AND IMPAIRMENT LOSSES FOR OTHER IN...    41899
265  BHCKC234  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDENTIAL PROPERTIES: SECURED BY FI...     4238
266  BHCKC217  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDENTIAL PROPERTIES: SECURED BY FI...     4230
267  BHCKC235  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDENTIAL PROPERTIES: SECURED BY JU...       22
268  BHCKC218  CLOSED-END LOANS SECURED BY 1-4 FAMILY RESIDENTIA

In [54]:
co_codes = {
    "BHCKC234": "1-4 family first lien",
    "BHCKC235": "1-4 family junior lien",
    "BHCK5411": "Home equity revolving",
    "BHCK3588": "Multifamily",
    "BHCKC891": "1-4 family construction",
    "BHCKC893": "Other construction and land",
    "BHCK3584": "Farmland",
    "BHCKC895": "CRE owner-occupied",
    "BHCKC897": "CRE other",
    "BHCK4645": "C&I US",
    "BHCK4646": "C&I non-US",
    "BHCK4655": "Agricultural",
    "BHCK4643": "Foreign governments",
    "BHCK4644": "All other loans",
    "BHCKC880": "All other leases",
    "BHCKF185": "Consumer leases",
    "BHCK4652": "RE non-US addressees",
}

co = y9c[y9c["ItemName"].isin(co_codes)].copy()
co["Value"] = pd.to_numeric(co["Value"])
co["Label"] = co["ItemName"].map(co_codes)

print(co[["ItemName", "Label", "Value"]].sort_values("Value", ascending=False).to_string(index=False))

total_co = val("BHCK4635")
print(f"\nSum of components: {co['Value'].sum():,}")
print(f"BHCK4635 total:    {total_co:,}")
print(f"Missing:           {total_co - co['Value'].sum():,}")

ItemName                       Label  Value
BHCK4645                      C&I US 282485
BHCKC897                   CRE other  82177
BHCK4644             All other loans  31182
BHCK3588                 Multifamily  26424
BHCKC880            All other leases  12837
BHCKC893 Other construction and land   6387
BHCKC234       1-4 family first lien   4238
BHCK5411       Home equity revolving   3863
BHCKC895          CRE owner-occupied   2000
BHCKC891     1-4 family construction    303
BHCKC235      1-4 family junior lien     22
BHCK4652        RE non-US addressees      0
BHCK4643         Foreign governments      0
BHCK3584                    Farmland      0
BHCK4646                  C&I non-US      0
BHCK4655                Agricultural      0
BHCKF185             Consumer leases      0

Sum of components: 451,918
BHCK4635 total:    746,153
Missing:           294,235


In [58]:
candidates = ["BHCKB514", "BHCKB515", "BHCKK133", "BHCKK134",
              "BHCKK205", "BHCKK206", "BHCKK216", "BHCKK217", "BHCKK218",
              "BHCKB575", "BHCKB576", "BHCKB577"]

cand = y9c[y9c["ItemName"].isin(candidates)].copy()
cand["Value"] = pd.to_numeric(cand["Value"], errors="coerce")
print(cand[["ItemName", "Description", "Value"]].to_string(index=False))

ItemName                                                                                                                                                                                              Description  Value
BHCKB514                                                                                 LOANS TO INDIVIDUALS FOR HOUSEHOLD, FAMILY, AND OTHER PERSONAL EXPENDITURES: CREDIT CARDS CHARGE-OFFS (BHC CONSOLIDATED)  46217
BHCKB515                                                                                  LOANS TO INDIVIDUALS FOR HOUSEHOLD, FAMILY, AND OTHER PERSONAL EXPENDITURES: CREDIT CARDS RECOVERIES (BHC CONSOLIDATED)   7601
BHCKB575                                                               LOANS TO INDIVIDUALS FOR HOUSEHOLDS, FAMILY AND OTHER PERSONAL EXPENDITURES: CREDIT CARDS PAST DUE 30-89 DAYS, ACCRUING (BHC CONSOLIDATED)  14597
BHCKB576                                                                 LOANS TO INDIVIDUALS FOR HOUSEHOLDS, FAMILY AND OTHER PERSO

In [59]:
print(y9c[y9c["ItemName"].str.startswith("BHCKK1", na=False)][["ItemName", "Description", "Value"]].head(30))

     ItemName  \
526  BHCKK141   
767  BHCKK129   
768  BHCKK133   
771  BHCKK142   
772  BHCKK143   
773  BHCKK144   
774  BHCKK145   
775  BHCKK146   
776  BHCKK147   
777  BHCKK148   
778  BHCKK149   
779  BHCKK150   
780  BHCKK151   
781  BHCKK152   
782  BHCKK153   
783  BHCKK154   
784  BHCKK155   
785  BHCKK156   
786  BHCKK157   
787  BHCKK137   
794  BHCKK163   
795  BHCKK164   
796  BHCKK165   
798  BHCKK168   
802  BHCKK197   
803  BHCKK198   
814  BHCKK192   
815  BHCKK193   
816  BHCKK194   
832  BHCKK102   

                                                                                                                                                                                                                                                                                                                                                                                        Description  \
526                                                                              

In [60]:
co_codes.update({
    "BHCKB514": "Consumer: Credit card",
    "BHCKK129": "Consumer: Automobile",
    "BHCKK205": "Consumer: Other",
})

co = y9c[y9c["ItemName"].isin(co_codes)].copy()
co["Value"] = pd.to_numeric(co["Value"])
co["Label"] = co["ItemName"].map(co_codes)

print(f"Sum of components: {co['Value'].sum():,}")
print(f"BHCK4635 total:    {val('BHCK4635'):,}")
print(f"Difference:        {co['Value'].sum() - val('BHCK4635'):,}")

Sum of components: 746,153
BHCK4635 total:    746,153
Difference:        0


In [61]:
rec_codes = {
    "BHCKC217": "1-4 family first lien",
    "BHCKC218": "1-4 family junior lien",
    "BHCK5412": "Home equity revolving",
    "BHCK3589": "Multifamily",
    "BHCKC892": "1-4 family construction",
    "BHCKC894": "Other construction and land",
    "BHCK3585": "Farmland",
    "BHCKC896": "CRE owner-occupied",
    "BHCKC898": "CRE other",
    "BHCK4617": "C&I US",
    "BHCK4618": "C&I non-US",
    "BHCKB515": "Consumer: Credit card",
    "BHCKK133": "Consumer: Automobile",
    "BHCKK206": "Consumer: Other",
    "BHCK4665": "Agricultural",
    "BHCK4627": "Foreign governments",
    "BHCK4628": "All other loans",
    "BHCKF188": "All other leases",
    "BHCKF187": "Consumer leases",
    "BHCK4662": "RE non-US addressees",
}

rec = y9c[y9c["ItemName"].isin(rec_codes)].copy()
rec["Value"] = pd.to_numeric(rec["Value"])

print(f"Sum of components: {rec['Value'].sum():,}")
print(f"BHCK4605 total:    {val('BHCK4605'):,}")
print(f"Difference:        {rec['Value'].sum() - val('BHCK4605'):,}")

Sum of components: 192,876
BHCK4605 total:    192,876
Difference:        0


In [62]:
print(find("NET INTEREST INCOME"))

      ItemName                                                    Description    Value
124   BHCK4074                         NET INTEREST INCOME (BHC CONSOLIDATED)  6948080
139   BHCK4519  NET INTEREST INCOME ON FULLY TAXABLE BASIS (BHC CONSOLIDATED)  6992233
984   BHBC4074                                            NET INTEREST INCOME        0
1002  BHBC4519                  NET INTEREST INCOME (FULLY TAXABLE EQV BASIS)        0


In [63]:
print(find("TOTAL NONINTEREST"))

     ItemName                                   Description    Value
128  BHCK4079   TOTAL NONINTEREST INCOME (BHC CONSOLIDATED)  2613465
132  BHCK4093  TOTAL NONINTEREST EXPENSE (BHC CONSOLIDATED)  5366094
985  BHBC4079                      TOTAL NONINTEREST INCOME        0


In [64]:
print(find("APPLICABLE INCOME TAXES"))

      ItemName                                                                                                                                             Description    Value
133   BHCK4301                                                     Income (loss) before applicable income taxes and discontinued operations (sum of items 8.a and 8.b)  3692210
134   BHCK4302                                                                                                              APPLICABLE INCOME TAXES (BHC CONSOLIDATED)   841227
140   BHCK4592                            Net income before applicable income taxes, and discontinued operations (item 8.c. above) on a fully taxable equivalent basis  3736362
997   BHBC4302                                                                                                                                 APPLICABLE INCOME TAXES        0
1492  BHCKFT41                                       DISCONTINUED OPERATIONS, NET OF APPLICABLE INCOME TAXES AND NONCONT

In [65]:
print(find("CASH DIVIDENDS"))

      ItemName                                                    Description   Value
144   BHCK4598  CASH DIVIDENDS DECLARED ON PREFERRED STOCK (BHC CONSOLIDATED)  146283
145   BHCK4460     CASH DIVIDENDS DECLARED ON COMMON STOCK (BHC CONSOLIDATED)  899637
1000  BHBC4475                                        CASH DIVIDENDS DECLARED       0
